# 05 - Coarse Classing & Feature Selection

Starting from the fine-classing spec of `04_woe_iv` (fit on train):

1. **Coarse classing**: merge adjacent numeric bins until the bad rate is strictly monotonic (the `Missing` bin stays separate). Monotonic WoE makes the scorecard stable and explainable.
2. **IV filter**: keep features with IV ≥ 0.02.
3. **Correlation filter**: visiting features by IV (best first), drop any feature whose WoE has |r| ≥ 0.7 with an already-kept feature.
4. **Stability check**: compare train vs test IV of the selected features.
5. Encode train and test with WoE for the modelling phase.

## 1. Setup

In [3]:
import pickle
import sys
from pathlib import Path

import pandas as pd

# Add project root to sys.path so `src` can be imported from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.selection import drop_correlated, select_by_iv
from src.woe import bin_frame, coarse_spec, is_monotonic, iv_summary, woe_frame, woe_table

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MIN_IV = 0.02
MAX_CORR = 0.7

## 2. Load data and fine-classing results

In [4]:
train = pd.read_pickle(DATA_DIR / "train.pkl")
test = pd.read_pickle(DATA_DIR / "test.pkl")

# Outputs of 04_woe_iv
with open(OUTPUT_DIR / "woe_spec.pkl", "rb") as f:
    spec_fine = pickle.load(f)
iv_fine = pd.read_csv(OUTPUT_DIR / "iv_summary.csv", index_col=0)

y_train, y_test = train["TARGET"], test["TARGET"]
print(f"Train: {train.shape} | Test: {test.shape} | Features: {len(spec_fine)}")

Train: (246008, 123) | Test: (61503, 123) | Features: 121


## 3. Coarse classing (monotonic merge)

In [5]:
# Merge numeric bins on train until bad rate is monotonic
spec = coarse_spec(train, y_train, spec_fine)

binned_train = bin_frame(train, spec)
tables = {col: woe_table(binned_train[col], y_train) for col in spec}
iv = iv_summary(tables)

# Every numeric feature must now be monotonic
numeric_cols = [c for c, rule in spec.items() if rule["type"] == "numeric"]
not_monotonic = [c for c in numeric_cols if not is_monotonic(tables[c])]
assert not not_monotonic, not_monotonic
print(f"OK: all {len(numeric_cols)} numeric features are monotonic")

OK: all 70 numeric features are monotonic


In [6]:
# IV kept after merging (some loss is expected; large loss = noisy fine bins)
compare = pd.DataFrame({
    "iv_fine": iv_fine["iv"],
    "iv_coarse": iv["iv"],
    "bins_fine": iv_fine["n_bins"],
    "bins_coarse": iv["n_bins"],
})
compare["iv_kept_pct"] = compare["iv_coarse"] / compare["iv_fine"] * 100
compare.sort_values("iv_coarse", ascending=False).head(25).round(4)

,iv_fine,iv_coarse,bins_fine,bins_coarse,iv_kept_pct
EXT_SOURCE_3,0.3296,0.3296,11,11,100.0000
EXT_SOURCE_2,0.3030,0.3030,11,11,100.0000
EXT_SOURCE_1,0.1512,0.1512,11,11,100.0000
YEARS_EMPLOYED,0.1116,0.1114,11,10,99.8756
AGE_YEARS,0.0867,0.0867,10,10,100.0000
OCCUPATION_TYPE,0.0754,0.0754,13,13,100.0000
ORGANIZATION_TYPE,0.0596,0.0596,18,18,100.0000
NAME_INCOME_TYPE,0.0558,0.0558,5,5,100.0000
REGION_RATING_CLIENT_W_CITY,0.0516,0.0516,3,3,100.0000
AMT_GOODS_PRICE,0.0913,0.0514,11,5,56.3210


## 4. IV filter

In [7]:
candidates = select_by_iv(iv, min_iv=MIN_IV)
print(f"{len(candidates)} / {len(iv)} features with IV >= {MIN_IV}")
iv["strength"].value_counts()

53 / 121 features with IV >= 0.02


strength
useless    68
weak       49
strong      2
medium      2
Name: count, dtype: int64

## 5. Correlation filter

In [8]:
# Correlation is measured on WoE values, the same scale the model will see
woe_train = woe_frame(binned_train, tables)
selected = drop_correlated(woe_train, candidates, threshold=MAX_CORR)
print(f"Selected {len(selected)} / {len(candidates)} candidates")

Selected 22 / 53 candidates


In [9]:
# Why each candidate was dropped: its most correlated kept feature
corr = woe_train[candidates].corr().abs()
dropped = [c for c in candidates if c not in selected]
pd.DataFrame(
    {
        "kept_instead": [corr.loc[c, selected].idxmax() for c in dropped],
        "abs_corr": [corr.loc[c, selected].max() for c in dropped],
        "iv": iv.loc[dropped, "iv"].values,
    },
    index=pd.Index(dropped, name="dropped"),
).round(3)

,kept_instead,abs_corr,iv
dropped,,,
REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,0.954,0.048
FLOORSMAX_MEDI,FLOORSMAX_AVG,0.997,0.038
FLOORSMAX_MODE,FLOORSMAX_AVG,0.988,0.037
TOTALAREA_MODE,FLOORSMAX_AVG,0.900,0.035
DAYS_EMPLOYED_ANOM,NAME_INCOME_TYPE,0.785,0.034
FLAG_EMP_PHONE,NAME_INCOME_TYPE,0.785,0.034
LIVINGAREA_AVG,FLOORSMAX_AVG,0.889,0.033
LIVINGAREA_MEDI,FLOORSMAX_AVG,0.888,0.032
APARTMENTS_AVG,FLOORSMAX_AVG,0.894,0.032


## 6. Selected features and stability (train vs test IV)

Test IV is computed with the **train** bins. A large drop means the binning overfits train.

In [10]:
binned_test = bin_frame(test, {c: spec[c] for c in selected})
iv_test = iv_summary({c: woe_table(binned_test[c], y_test) for c in selected})

final = iv.loc[selected, ["iv", "n_bins", "strength"]].rename(columns={"iv": "iv_train"})
final["iv_test"] = iv_test["iv"]
final["test_vs_train_pct"] = final["iv_test"] / final["iv_train"] * 100
final.round(4)

,iv_train,n_bins,strength,iv_test,test_vs_train_pct
EXT_SOURCE_3,0.3296,11,strong,0.3310,100.4489
EXT_SOURCE_2,0.3030,11,strong,0.3207,105.8366
EXT_SOURCE_1,0.1512,11,medium,0.1489,98.4951
YEARS_EMPLOYED,0.1114,10,medium,0.1136,101.9577
AGE_YEARS,0.0867,10,weak,0.0758,87.4064
OCCUPATION_TYPE,0.0754,13,weak,0.0815,108.1142
ORGANIZATION_TYPE,0.0596,18,weak,0.0539,90.3981
NAME_INCOME_TYPE,0.0558,5,weak,0.0666,119.3208
REGION_RATING_CLIENT_W_CITY,0.0516,3,weak,0.0499,96.7710
AMT_GOODS_PRICE,0.0514,5,weak,0.0495,96.2729


## 7. Inspect coarse WoE tables

In [11]:
# Fine bins 1-2 were reversed; they should now be merged
tables["YEARS_EMPLOYED"].round(4)

,n,share,n_bad,bad_rate,woe,iv
bin,,,,,,
"(-inf, 1.69]",40638,0.1652,4593,0.1130,-0.3721,0.0267
"(1.69, 2.51]",19966,0.0812,2194,0.1099,-0.3406,0.0109
"(2.51, 3.42]",20207,0.0821,2082,0.1030,-0.2685,0.0066
"(3.42, 4.51]",20215,0.0822,1910,0.0945,-0.1724,0.0026
"(4.51, 5.91]",20153,0.0819,1696,0.0842,-0.0453,0.0002
"(5.91, 7.64]",20140,0.0819,1528,0.0759,0.0673,0.0004
"(7.64, 10.07]",20234,0.0822,1405,0.0694,0.1628,0.0020
"(10.07, 14.62]",20161,0.0820,1182,0.0586,0.3435,0.0084
"(14.62, inf]",20151,0.0819,900,0.0447,0.6301,0.0251


In [12]:
# More occupations keep their own bin with min_share = 1%
tables["OCCUPATION_TYPE"].round(4)

,n,share,n_bad,bad_rate,woe,iv
bin,,,,,,
Accountants,7882,0.0320,365,0.0463,0.5915,0.0088
Cleaning staff,3741,0.0152,357,0.0954,-0.1844,0.0006
Cooking staff,4767,0.0194,501,0.1051,-0.2912,0.0019
Core staff,22093,0.0898,1410,0.0638,0.2532,0.0052
Drivers,14944,0.0607,1683,0.1126,-0.3682,0.0096
High skill tech staff,9093,0.0370,571,0.0628,0.2700,0.0024
Laborers,44259,0.1799,4657,0.1052,-0.2918,0.0173
Managers,17096,0.0695,1064,0.0622,0.2799,0.0048
Medicine staff,6871,0.0279,480,0.0699,0.1557,0.0006


In [13]:
tables["ORGANIZATION_TYPE"].round(4)

,n,share,n_bad,bad_rate,woe,iv
bin,,,,,,
Business Entity Type 1,4820,0.0196,388,0.0805,0.0024,0.0000
Business Entity Type 2,8431,0.0343,728,0.0863,-0.0736,0.0002
Business Entity Type 3,54436,0.2213,5048,0.0927,-0.1514,0.0054
Construction,5399,0.0219,645,0.1195,-0.4352,0.0050
Government,8341,0.0339,573,0.0687,0.1740,0.0010
Industry: type 3,2650,0.0108,286,0.1079,-0.3215,0.0013
Industry: type 9,2720,0.0111,181,0.0665,0.2064,0.0004
Kindergarten,5552,0.0226,396,0.0713,0.1333,0.0004
Medicine,8985,0.0365,606,0.0674,0.1938,0.0013


## 8. WoE-encode and save

In [14]:
# Test uses train bins and train WoE values (no leakage)
selected_tables = {c: tables[c] for c in selected}
woe_test = woe_frame(binned_test, selected_tables)

woe_train[selected].assign(TARGET=y_train).to_pickle(DATA_DIR / "woe_train.pkl")
woe_test.assign(TARGET=y_test).to_pickle(DATA_DIR / "woe_test.pkl")

with open(OUTPUT_DIR / "woe_spec_final.pkl", "wb") as f:
    pickle.dump({c: spec[c] for c in selected}, f)
with open(OUTPUT_DIR / "woe_tables_final.pkl", "wb") as f:
    pickle.dump(selected_tables, f)
final.to_csv(OUTPUT_DIR / "selected_features.csv")

print("Saved woe_train.pkl, woe_test.pkl, woe_spec_final.pkl, woe_tables_final.pkl, selected_features.csv")

Saved woe_train.pkl, woe_test.pkl, woe_spec_final.pkl, woe_tables_final.pkl, selected_features.csv


## 9. Review: missing-value decisions

EDA suggested a strategy from the missing rate alone; here the final keep/drop decision is based on IV. Columns with a high missing rate can still be selected.

In [15]:
from src.eda import missing_summary, suggest_missing_strategy

review = missing_summary(train)[["pct_missing"]].join(iv[["iv", "strength"]])
review["eda_strategy"] = review["pct_missing"].apply(suggest_missing_strategy)
review["decision"] = [
    "selected" if c in selected
    else "dropped: correlated" if c in candidates
    else "dropped: low IV"
    for c in review.index
]

# EDA suggestion vs. final decision
pd.crosstab(review["eda_strategy"], review["decision"])

decision,dropped: correlated,dropped: low IV,selected
eda_strategy,,,
drop_candidate,0,16,1
impute,0,7,5
missing_bin,27,9,5


In [16]:
# Missing columns that were finally selected
review[review["decision"] == "selected"].round(3)

,pct_missing,iv,strength,eda_strategy,decision
OWN_CAR_AGE,66.021,0.022,weak,drop_candidate,selected
EXT_SOURCE_1,56.338,0.151,medium,missing_bin,selected
FLOORSMAX_AVG,49.713,0.038,weak,missing_bin,selected
OCCUPATION_TYPE,31.275,0.075,weak,missing_bin,selected
EXT_SOURCE_3,19.839,0.330,strong,missing_bin,selected
YEARS_EMPLOYED,17.944,0.111,medium,missing_bin,selected
EXT_SOURCE_2,0.216,0.303,strong,impute,selected
AMT_GOODS_PRICE,0.090,0.051,weak,impute,selected
NAME_FAMILY_STATUS,0.001,0.023,weak,impute,selected
CODE_GENDER,0.001,0.038,weak,impute,selected


In [17]:
import pandas as pd

woe_train = pd.read_pickle("../data/woe_train.pkl")

print(woe_train.shape)        # (246008, 23): 22 WoE features + TARGET
woe_train.head(10).round(3)

(246008, 23)


,EXT_SOURCE_3,EXT_SOURCE_2,EXT_SOURCE_1,YEARS_EMPLOYED,AGE_YEARS,OCCUPATION_TYPE,ORGANIZATION_TYPE,NAME_INCOME_TYPE,REGION_RATING_CLIENT_W_CITY,AMT_GOODS_PRICE,...,FLOORSMAX_AVG,YEARS_ID_PUBLISH,REG_CITY_NOT_WORK_CITY,REGION_POPULATION_RELATIVE,FLAG_DOCUMENT_3,YEARS_REGISTRATION,REG_CITY_NOT_LIVE_CITY,NAME_FAMILY_STATUS,OWN_CAR_AGE,TARGET
181648,0.545,-0.449,-0.049,-0.269,-0.214,-0.292,-0.074,0.067,0.023,-0.172,...,0.299,0.282,0.109,-0.112,-0.100,0.024,0.050,0.075,-0.054,0
229245,-0.152,0.085,-0.059,-0.372,0.004,-0.368,0.081,0.067,-0.388,-0.172,...,-0.141,-0.144,-0.305,-0.112,-0.100,-0.110,0.050,0.075,-0.157,0
122525,0.545,-0.045,-0.059,-0.172,0.265,-0.292,-0.151,-0.183,0.023,0.048,...,0.416,-0.050,0.109,-0.088,-0.100,0.124,0.050,-0.229,-0.054,1
306311,0.018,0.506,-0.059,-0.045,0.147,-0.292,-0.151,0.067,0.023,-0.172,...,-0.158,-0.209,0.109,-0.088,-0.100,-0.110,0.050,-0.229,-0.054,1
300658,-0.152,-0.935,-0.059,-0.269,-0.406,0.592,0.060,0.067,0.023,-0.172,...,-0.141,-0.209,0.109,-0.112,-0.100,-0.078,0.050,-0.229,-0.054,0
201033,0.545,0.345,-0.059,0.437,0.539,0.235,0.437,0.439,0.023,-0.172,...,0.527,0.059,0.109,0.163,0.287,0.387,0.050,0.075,-0.054,0
86445,-0.511,0.345,-0.059,0.067,0.004,-0.368,0.060,-0.183,0.023,-0.172,...,0.125,0.221,0.109,-0.088,-0.100,-0.110,0.050,-0.237,0.385,0
225267,-0.152,0.085,0.326,-0.172,-0.099,-0.190,-0.258,0.067,0.023,0.522,...,-0.141,0.221,0.109,-0.088,-0.100,0.387,-0.473,0.075,-0.054,0
42927,0.018,0.345,-0.875,-0.172,-0.406,-0.190,0.081,0.067,0.023,-0.172,...,0.299,-0.105,0.109,0.081,-0.100,-0.110,0.050,-0.237,-0.054,0
240820,0.545,-0.449,-0.059,-0.372,0.432,0.270,0.060,0.067,0.023,-0.172,...,-0.141,0.321,-0.305,0.013,0.287,-0.110,0.050,0.075,-0.096,0
